# Final Model Evaluation

## 1. Objective

Evaluate the selected Ridge model (alpha = 10) on the held-out test set. Ridge was selected using the lowest mean cross-validation RMSE among Linear Regression, Ridge, and Lasso.

The saved pipeline was fitted on the full training dataset. Test data will only be transformed and predicted using this fitted pipeline; no preprocessing or model parameters will be fitted on the test set.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

In [2]:
project_root = Path.cwd().parent

X_test = pd.read_csv(
    project_root / "data/processed/X_test.csv",
    index_col="row_index",
    dtype={"Postcode": str}
)

y_test = pd.read_csv(
    project_root / "data/processed/y_test.csv",
    index_col="row_index"
)["Price"]

final_pipeline = joblib.load(
    project_root / "models/ridge_pipeline_fitted.joblib"
)

assert X_test.index.equals(y_test.index)
assert X_test.index.is_unique
assert y_test.notna().all()

assert list(X_test.columns) == list(
    final_pipeline.feature_names_in_
)

print("Test predictors:", X_test.shape)
print("Test target:", y_test.shape)
print("Selected alpha:", final_pipeline.named_steps["model"].alpha)

Test predictors: (2716, 16)
Test target: (2716,)
Selected alpha: 10


## 2. Evaluate on the Held-Out Test Set

Generate predictions using the saved Ridge pipeline and calculate MAE, RMSE, and R². The model and preprocessing remain fixed during evaluation.

In [3]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

test_predictions = final_pipeline.predict(X_test)

assert np.isfinite(test_predictions).all()

test_metrics = pd.DataFrame({
    "MAE": [mean_absolute_error(y_test, test_predictions)],
    "RMSE": [np.sqrt(mean_squared_error(y_test, test_predictions))],
    "R2": [r2_score(y_test, test_predictions)]
}, index=["Ridge"])

test_metrics.index.name = "Model"
test_metrics.round(3)

,MAE,RMSE,R2
Model,,,
Ridge,233860.352,349213.556,0.693


### Observation

The selected Ridge model achieved a held-out test MAE of 233,860, RMSE of 349,214, and R² of 0.693 across 2,716 properties. MAE represents an average absolute prediction error of approximately AUD 234,000; R² should not be interpreted as percentage accuracy.

Test performance was better than the mean cross-validation results (MAE 237,848, RMSE 388,385, and R² 0.631). Differences in sample composition and extreme observations can affect these metrics, so this single split does not establish consistently better performance on future data. No model settings were changed using the test results.

## 3. Save Final Evaluation Results

Save held-out test metrics and row-level predictions with signed residuals and absolute errors for reproducibility.

In [4]:
tables_dir = project_root / "results/tables"
tables_dir.mkdir(parents=True, exist_ok=True)

test_results = pd.DataFrame({
    "Actual_Price": y_test,
    "Predicted_Price": test_predictions
})

test_results["Residual"] = (
    test_results["Actual_Price"] - test_results["Predicted_Price"]
)

test_results["Absolute_Error"] = test_results["Residual"].abs()

test_metrics.to_csv(
    tables_dir / "ridge_final_test_metrics.csv"
)

test_results.to_csv(
    tables_dir / "ridge_final_test_predictions.csv",
    index_label="row_index"
)

print("Final test metrics and predictions saved.")

Final test metrics and predictions saved.


## 4. Final Conclusion and Limitations

This project compared Linear Regression, Ridge, and Lasso using the same training data, preprocessing, and five cross-validation folds. Ridge with alpha = 10 achieved the lowest mean validation RMSE and was selected before evaluating the held-out test set.

On 2,716 held-out properties, Ridge achieved MAE of 233,860, RMSE of 349,214, and R² of 0.693. These results provide a final evaluation of the selected pipeline on the reserved random split.

Important limitations remain. Extreme recorded property areas and prices were retained where their validity could not be established. Training OOF diagnostics showed substantial errors for some properties, particularly expensive homes. A random train-test split does not establish performance on future sales or unseen geographic markets.

The saved pipeline includes fitted imputation, scaling, categorical encoding, and Ridge regression. It expects the 16 prepared predictor columns; raw-data cleaning rules must be applied separately.

Further model development should use training cross-validation and a new independent evaluation set rather than repeatedly tuning against this test set.